# V2 Control Panel

**Interactive control surface for the frozen V2 "neuron as vector" architecture.**
It is a thin interface over the stable backend - every operation goes through
`src/v2_pipeline.py`, which is also what the headless CLI (`scripts/v2_control_panel.py`) uses.
No representation logic, no metric and no fitting lives in this notebook.

```
Jupyter notebook (this file)
        |
        v
V2Config  (src/v2_config.py - the single configuration system)
        |
        v
src/v2_pipeline.py   ->  build (FIT only)  |  inspect cache  |  evaluate (PROBE only)
        |
        v
NeuronRecordBank -> StructuredVectorEncoder (+ learned residual) -> NeuronVector (n_neurons, d)
```

| interface | role |
|---|---|
| **this notebook** | primary interactive interface: configure, validate, build, inspect, evaluate |
| `scripts/v2_control_panel.py` | headless / reproducible interface (same backend, same run ids) |
| `src/v2_pipeline.py` | the stable backend (`V2Run`, `resolve_config`, `build_representation`, …) |

**How to use it** - run the sections from top to bottom, in order:

1. environment / project initialization (no data is read)
2. pick a preset, then change one or two controls
3. **validate** and read the resolved-configuration preview
4. **build / load** the representation (the only expensive step; FIT only)
5. **inspect** the frozen artifact
6. optionally **evaluate** it on PROBE (a separate, explicit step)

Nothing here installs packages, touches TEST, or runs a scientific sweep. The heavy steps are
isolated in their own cells so nothing expensive happens by accident.

*Requirements: any Jupyter kernel (Python 3.11) with this repository's environment.*
Interference-free: `ipywidgets` is optional - if it is unavailable, each control is an ordinary
editable variable (see the optional-widgets cell).

## 1. Environment / project initialization

Finds the repository root, imports the backend, reports versions and confirms that the V2
configuration system loads. **No dataset is read in this cell.**

Panel settings (config file, checkpoint, artifact directory) are ordinary editable variables;
environment variables `V2_PANEL_CONFIG`, `V2_PANEL_CHECKPOINT` and `V2_PANEL_OUT_DIR` override
them, which is how the automated notebook smoke tests run this notebook headlessly.

In [1]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk upwards until the repository markers are found (works from any working directory)."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "v2_pipeline.py").exists():
            return candidate
    raise RuntimeError("could not locate the repository root (pyproject.toml + src/v2_pipeline.py)")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch

from src import v2_panel as panel                     # notebook-facing layer (state -> config -> pipeline)
from src.v2_pipeline import DATA_POLICY, SCHEMA, available_presets, parent_summary

# -- panel settings (editable; environment variables allow headless use) ---------
CONFIG_PATH = os.environ.get("V2_PANEL_CONFIG", "configs/neuron_space_baseline.yaml")
OUT_DIR = os.environ.get("V2_PANEL_OUT_DIR", "results/neuron_vector_capacity")
CHECKPOINT = (
    os.environ.get("V2_PANEL_CHECKPOINT")
    or panel.default_state(config_path=CONFIG_PATH)["checkpoint"]      # the config's checkpoint
)

print(f"python       : {sys.version.split()[0]}  ({sys.executable})")
print(f"repo root    : {ROOT}")
print(f"numpy / torch: {np.__version__} / {torch.__version__}  (cuda available: {torch.cuda.is_available()})")
print(f"schemas      : pipeline={SCHEMA}  panel_state={panel.PANEL_SCHEMA}")
widgets = panel.widget_support()
print(f"ipywidgets   : {'available' if widgets['available'] else 'not available -> ' + widgets['fallback']}")
print(f"config file  : {CONFIG_PATH}")
print(f"checkpoint   : {CHECKPOINT}")
print(f"artifacts    : {OUT_DIR}")
print(f"presets      : {', '.join(available_presets())}")
print(f"controls     : {len(panel.panel_controls())} in groups {panel.CONTROL_GROUPS}")

python       : 3.11.14  (c:\Users\Administrator\Desktop\compneuro\neuron_as_vector\.venv\Scripts\python.exe)
repo root    : C:\Users\Administrator\Desktop\compneuro\neuron_as_vector
numpy / torch: 2.4.6 / 2.14.0+cu126  (cuda available: True)
schemas      : pipeline=v2_pipeline/v1  panel_state=v2_panel_state/v1
ipywidgets   : not available -> use the clearly marked editable variables in each cell
config file  : configs/neuron_space_baseline.yaml
checkpoint   : C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\checkpoints\sweep_l2_0.pt
artifacts    : results/neuron_vector_capacity
presets      : historical_48, structured_64, structured_100, temporal_structured, functional_64, functional_100
controls     : 47 in groups ('panel', 'representation', 'blocks', 'functional_response', 'temporal', 'residual', 'model', 'memory', 'precision', 'evaluation')


## 2. Presets (asked from the backend)

The preset list is read from `src/v2_pipeline.PRESETS` - a preset added there appears here
automatically. A preset is *configuration*, never a hidden execution path: each one pins the
split (`structured_d` + `learned_residual_d`) and the source switches, and the total `d` is
derived from it.

In [2]:
print("available presets")
print("=================")
for name in available_presets():
    print(f"  {name:20s} d = {panel.PRESET_DECOMPOSITIONS[name]}")
    print(f"  {'':20s} {panel.PRESET_DESCRIPTIONS[name]}")

# ---------------------------------------------------------------------------
PRESET = "historical_48"      # <- EDITABLE: any preset name above, or None for the current default
# ---------------------------------------------------------------------------

available presets
  historical_48        d = 48 = 48 structured + 0 residual (default structural blocks; regression anchor)
                       the historical 48-D structural representation (bit-identical anchor)
  structured_64        d = 64 = 64 structured + 0 residual (level-0 48 + first 16 level-1 coordinates)
                       a wider purely deterministic structural vector
  structured_100       d = 100 = 100 structured + 0 residual (level-0 48 + 52 level-1 coordinates)
                       the widest purely deterministic structural vector before projection
  temporal_structured  d = 58 = 58 structured + 0 residual (default blocks + 10 coarse temporal bins)
                       the deterministic structural vector plus coarse temporal bins
  functional_64        d = 64 = 48 structured + 16 residual (residual consumes structural + functional-response)
                       the source-extension arm B decomposition at residual_d = 16
  functional_100       d = 100 = 48 st

In [3]:
state = panel.default_state(
    preset=PRESET,
    config_path=CONFIG_PATH,
    checkpoint=CHECKPOINT,
    out_dir=OUT_DIR,
)
print(f"state created for preset: {PRESET!r}")
print(f"  d = {state['d']} = {state['structured_d']} structured + {state['learned_residual_d']} residual")
print(f"  encoder blocks: {state['enabled_blocks']}")
print(f"  residual enabled: {state['residual_enabled']}  (sources: "
      f"functional_response={state['source_functional_response']}, temporal={state['source_temporal']})")


def sync_panel_settings() -> None:
    """Push the editable panel settings (cell 1) into the state - called by every action cell."""
    panel.set_controls(
        state,
        config_path=str(CONFIG_PATH),
        preset=PRESET,
        checkpoint=str(CHECKPOINT),
        out_dir=str(OUT_DIR),
    )


sync_panel_settings()

state created for preset: 'historical_48'
  d = 48 = 48 structured + 0 residual
  encoder blocks: ['intrinsic', 'input_conn', 'recurrent_in', 'recurrent_out']
  residual enabled: False  (sources: functional_response=False, temporal=False)


In [4]:
# Optional: interactive widgets. They are used only when ipywidgets is importable; the editable
# variables in each cell below always work, so the notebook never depends on a widget toolkit.
support = panel.widget_support()
if support["available"]:
    form = panel.make_widgets(state, groups=("representation", "blocks", "functional_response",
                                            "temporal", "residual", "memory"))
    print("widget form created - changing a widget updates `state` in place.")
else:
    print(f"no widgets in this environment: {support['reason']}")
    print(f"fallback: {support['fallback']}")
    form = None

no widgets in this environment: ipywidgets is not importable (ModuleNotFoundError: No module named 'ipywidgets')
fallback: use the clearly marked editable variables in each cell


## 3. Representation dimensions

The invariant is **`d = structured_d + residual_d`**. `d` is *derived* and never stored, so it
cannot disagree with the split; the checks below report the equation and any combination that the
backend would reject (residual capacity without an enabled residual, sources without a residual).

In [5]:
# ---------------------------------------------------------------------------
STRUCTURED_D = state["structured_d"]                 # <- EDITABLE: deterministic coordinates
LEARNED_RESIDUAL_D = state["learned_residual_d"]     # <- EDITABLE: 0 = no learned residual
# ---------------------------------------------------------------------------
# To specify the total directly instead: panel.set_controls(state, structured_d=TOTAL_D,
# learned_residual_d=0) - the equation then holds by construction.

try:
    panel.set_controls(state, structured_d=STRUCTURED_D, learned_residual_d=LEARNED_RESIDUAL_D)
except panel.PanelError as exc:
    print("[invalid] ", exc)

check = panel.dimension_check(state)
print("d = structured_d + residual_d")
print(f"    {check['d']} = {check['structured_d']} + {check['residual_d']}")
print("  " + ("OK" if check["ok"] else "[INVALID] " + "; ".join(check["problems"])))

d = structured_d + residual_d
    48 = 48 + 0
  OK


## 4. Structural blocks

Only **implemented** blocks are selectable (the backend's `IMPLEMENTED_RECORD_BLOCKS`).
`network_context` is declared but not implemented, so it is displayed and not offered.
The default selection stays the historical four structural blocks.

In [6]:
print("selectable blocks   :", ", ".join(panel.SELECTABLE_BLOCKS))
print("not selectable      : network_context = not implemented")
print("display-only fields :", ", ".join(panel.DISPLAY_ONLY))

# ---------------------------------------------------------------------------
ENABLED_BLOCKS = list(state["enabled_blocks"])      # <- EDITABLE (list, or "a,b,c" as a string)
# ---------------------------------------------------------------------------
try:
    panel.set_controls(state, enabled_blocks=ENABLED_BLOCKS)
except panel.PanelError as exc:
    print("[invalid] ", exc)
print("enabled blocks (structured encoder):", state["enabled_blocks"])
print("temporal block requested in the bank:", "temporal" in state["enabled_blocks"] or state["source_temporal"])

selectable blocks   : intrinsic, input_conn, recurrent_in, recurrent_out, activity, temporal
not selectable      : network_context = not implemented
display-only fields : d, n_layers, network_context
enabled blocks (structured encoder): ['intrinsic', 'input_conn', 'recurrent_in', 'recurrent_out']
temporal block requested in the bank: False


## 5. Functional-response source

Label-free source view of the FIT per-stimulus response profile (a fixed, seeded projection).
It is an input of the **learned residual**, so it requires `residual_enabled` with
`learned_residual_d > 0`. Supported values come from the backend
(`FUNCTIONAL_SOURCE_NORMALIZATIONS`).

In [7]:
print("supported normalizations:", ", ".join(panel.FUNCTIONAL_SOURCE_NORMALIZATIONS))
print("projection dimension bound:", panel.MAX_FUNCTIONAL_SOURCE_DIM)

# ---------------------------------------------------------------------------
FUNCTIONAL_RESPONSE = state["source_functional_response"]          # <- EDITABLE (bool)
FUNCTIONAL_SOURCE_DIM = state["functional_source_dim"]             # <- EDITABLE
FUNCTIONAL_PROJECTION_SEED = state["functional_projection_seed"]   # <- EDITABLE
FUNCTIONAL_NORMALIZATION = state["functional_source_normalization"]  # <- EDITABLE (see above)
# ---------------------------------------------------------------------------
try:
    panel.set_controls(
        state,
        source_functional_response=FUNCTIONAL_RESPONSE,
        functional_source_dim=FUNCTIONAL_SOURCE_DIM,
        functional_projection_seed=FUNCTIONAL_PROJECTION_SEED,
        functional_source_normalization=FUNCTIONAL_NORMALIZATION,
    )
except panel.PanelError as exc:
    print("[invalid] ", exc)

print(f"functional_response (residual source): {state['source_functional_response']}")
print(f"  dim={state['functional_source_dim']}  projection_seed={state['functional_projection_seed']}  "
      f"normalization={state['functional_source_normalization']}")

supported normalizations: raw, neuron_centered, neuron_zscored
projection dimension bound: 4096
functional_response (residual source): False
  dim=64  projection_seed=0  normalization=raw


## 6. Temporal source

The coarse label-free FIT temporal block. It can be selected by the *structured encoder*
(via the blocks cell) and/or consumed by the *learned residual* here; the residual switch is
independent of the encoder selection.

In [8]:
# ---------------------------------------------------------------------------
TEMPORAL_SOURCE = state["source_temporal"]           # <- EDITABLE (residual consumes the block)
TEMPORAL_RESOLUTION = state["temporal_resolution"]   # <- EDITABLE (coarse FIT PSTH bins)
# ---------------------------------------------------------------------------
try:
    panel.set_controls(state, source_temporal=TEMPORAL_SOURCE, temporal_resolution=TEMPORAL_RESOLUTION)
except panel.PanelError as exc:
    print("[invalid] ", exc)

print(f"temporal (residual source): {state['source_temporal']}")
print(f"temporal_resolution       : {state['temporal_resolution']}")
print(f"encoder selects temporal  : {'temporal' in state['enabled_blocks']}")

temporal (residual source): False
temporal_resolution       : 10
encoder selects temporal  : False


## 7. Residual controls

The learned residual is the only trained component (FIT-only, self-supervised). Its **established
protocol** is 200 epochs, batch 64, Adam `lr=1e-3`, hidden 64, `mask_fraction=0.25`, coordinate
masking, `val_fraction=0.2`, `train_standardise`. The cell prints the **drift** whenever a value
differs from that protocol, so a modified protocol can never go unnoticed.

In [9]:
# ---------------------------------------------------------------------------
RESIDUAL_ENABLED = state["residual_enabled"]                  # <- EDITABLE
RESIDUAL_SEED = state["residual_seed"]                        # <- EDITABLE
RESIDUAL_SPLIT_SEED = state["residual_split_seed"]             # <- EDITABLE
RESIDUAL_MASK_SEED = state["residual_mask_seed"]               # <- EDITABLE
RESIDUAL_HIDDEN_DIM = state["residual_hidden_dim"]             # <- EDITABLE
RESIDUAL_EPOCHS = state["residual_epochs"]                     # <- EDITABLE
RESIDUAL_BATCH_SIZE = state["residual_batch_size"]             # <- EDITABLE
RESIDUAL_LEARNING_RATE = state["residual_learning_rate"]       # <- EDITABLE
RESIDUAL_MASK_FRACTION = state["residual_mask_fraction"]       # <- EDITABLE
RESIDUAL_MIN_VISIBLE = state["residual_minimum_visible_features"]  # <- EDITABLE
RESIDUAL_VAL_FRACTION = state["residual_val_fraction"]         # <- EDITABLE
RESIDUAL_STANDARDIZATION = state["residual_standardization"]   # <- EDITABLE
RESIDUAL_MASK_MODE = state["residual_mask_mode"]               # <- EDITABLE
# ---------------------------------------------------------------------------
try:
    panel.set_controls(
        state,
        residual_enabled=RESIDUAL_ENABLED,
        residual_seed=RESIDUAL_SEED,
        residual_split_seed=RESIDUAL_SPLIT_SEED,
        residual_mask_seed=RESIDUAL_MASK_SEED,
        residual_hidden_dim=RESIDUAL_HIDDEN_DIM,
        residual_epochs=RESIDUAL_EPOCHS,
        residual_batch_size=RESIDUAL_BATCH_SIZE,
        residual_learning_rate=RESIDUAL_LEARNING_RATE,
        residual_mask_fraction=RESIDUAL_MASK_FRACTION,
        residual_minimum_visible_features=RESIDUAL_MIN_VISIBLE,
        residual_val_fraction=RESIDUAL_VAL_FRACTION,
        residual_standardization=RESIDUAL_STANDARDIZATION,
        residual_mask_mode=RESIDUAL_MASK_MODE,
    )
except panel.PanelError as exc:
    print("[invalid] ", exc)

drift = panel.protocol_drift(state)
print(f"residual_enabled: {state['residual_enabled']}   residual_d: {state['learned_residual_d']}")
print(f"epochs={state['residual_epochs']}  batch={state['residual_batch_size']}  "
      f"lr={state['residual_learning_rate']}  hidden={state['residual_hidden_dim']}")
print(f"mask: mode={state['residual_mask_mode']}  fraction={state['residual_mask_fraction']}  "
      f"mask_seed={state['residual_mask_seed']}  min_visible={state['residual_minimum_visible_features']}")
print(f"split: val_fraction={state['residual_val_fraction']}  split_seed={state['residual_split_seed']}  "
      f"seed={state['residual_seed']}  standardization={state['residual_standardization']}")
print("protocol drift vs the established protocol:",
      "none" if not drift else "; ".join(f"{k}: {v[0]!r} != default {v[1]!r}" for k, v in drift.items()))

residual_enabled: False   residual_d: 0
epochs=200  batch=64  lr=0.001  hidden=64
mask: mode=coordinate  fraction=0.25  mask_seed=0  min_visible=8
split: val_fraction=0.2  split_seed=0  seed=0  standardization=train_standardise
protocol drift vs the established protocol: none


## 8. Compute / memory / precision

The existing V2 controls - no notebook-only knobs. Values that the pipeline cannot honour
(`storage != cpu`, non-`float32` `model_dtype`, `mixed_precision`) are rejected or reported as
warnings by the backend, never silently applied.

In [10]:
print("devices :", ", ".join(panel.SUPPORTED_DEVICES))
print("storage :", ", ".join(panel.SUPPORTED_STORAGE))
print("dtypes  :", ", ".join(panel.SUPPORTED_DTYPES))

# ---------------------------------------------------------------------------
DEVICE = state["device"]                       # <- EDITABLE
STORAGE = state["storage"]                     # <- EDITABLE
TRAIN_BATCH_SIZE = state["train_batch_size"]   # <- EDITABLE
EVAL_BATCH_SIZE = state["eval_batch_size"]     # <- EDITABLE
RECORD_BATCH_SIZE = state["record_batch_size"] # <- EDITABLE
ACTIVITY_CHUNK_SIZE = state["activity_chunk_size"]              # <- EDITABLE
REPRESENTATION_CHUNK_SIZE = state["representation_chunk_size"]  # <- EDITABLE
VECTOR_DTYPE = state["vector_dtype"]           # <- EDITABLE
ACTIVITY_DTYPE = state["activity_dtype"]       # <- EDITABLE
MODEL_DTYPE = state["model_dtype"]             # <- EDITABLE
MIXED_PRECISION = state["mixed_precision"]     # <- EDITABLE (not implemented; rejected)
MAX_INPUT_TOKENS = state["max_input_tokens"]   # <- EDITABLE (0 disables the guard)
# ---------------------------------------------------------------------------
try:
    panel.set_controls(
        state,
        device=DEVICE, storage=STORAGE,
        train_batch_size=TRAIN_BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE,
        record_batch_size=RECORD_BATCH_SIZE, activity_chunk_size=ACTIVITY_CHUNK_SIZE,
        representation_chunk_size=REPRESENTATION_CHUNK_SIZE,
        vector_dtype=VECTOR_DTYPE, activity_dtype=ACTIVITY_DTYPE, model_dtype=MODEL_DTYPE,
        mixed_precision=MIXED_PRECISION, max_input_tokens=MAX_INPUT_TOKENS,
    )
except panel.PanelError as exc:
    print("[invalid] ", exc)

for name in ("device", "storage", "train_batch_size", "eval_batch_size", "record_batch_size",
             "activity_chunk_size", "representation_chunk_size", "vector_dtype", "activity_dtype",
             "model_dtype", "mixed_precision", "max_input_tokens"):
    print(f"  {name:26s} {state[name]}")

devices : auto, cpu, cuda
storage : cpu, gpu, memmap
dtypes  : float32, float16, bfloat16
  device                     auto
  storage                    cpu
  train_batch_size           128
  eval_batch_size            256
  record_batch_size          32
  activity_chunk_size        32
  representation_chunk_size  256
  vector_dtype               float32
  activity_dtype             float32
  model_dtype                float32
  mixed_precision            False
  max_input_tokens           150000000


## 9. Checkpoint

The notebook enumerates the available checkpoints itself and loads the selected one to report its
architecture and identifier. `model.n_hidden` / `model.n_bins` must match the checkpoint - the
backend rejects a mismatch, and the compatibility check below shows it before any expensive step.

In [11]:
candidates = panel.checkpoint_candidates()
print(f"{len(candidates)} checkpoints found (default: "
      f"{[c['name'] for c in candidates if c['is_default']] or 'none'})")
for item in candidates[:10]:
    print(f"  {item['name']:30s} {item['size_mb']:5.2f} MB")
if len(candidates) > 10:
    print(f"  ... {len(candidates) - 10} more")

# ---------------------------------------------------------------------------
CHECKPOINT = str(CHECKPOINT)     # <- EDITABLE: any path printed above (or an absolute path)
# ---------------------------------------------------------------------------

details = panel.checkpoint_details(CHECKPOINT)
print("\nselected checkpoint:", details["path"])
print("  sha256_16:", details["sha256_16"])
for key, value in details["architecture"].items():
    print(f"  {key:20s} {value}")
compatibility = panel.checkpoint_mismatches(CHECKPOINT, CONFIG_PATH)
print("  config compatibility:", "OK" if compatibility["compatible"] else compatibility["mismatches"])
sync_panel_settings()

18 checkpoints found (default: ['sweep_l2_0.pt'])
  baseline.pt                     1.21 MB
  baseline_repaired_v1.pt         1.21 MB
  baseline_s1.pt                  1.21 MB
  baseline_s2.pt                  1.21 MB
  baseline_v2.pt                  1.21 MB
  nsb_seed1.pt                    1.21 MB
  nsb_seed2.pt                    1.21 MB
  runcheck.pt                     1.21 MB
  sweep_bin2_n500_1000ms.pt       1.21 MB
  sweep_bin2_n700_1400ms.pt       1.21 MB
  ... 8 more

selected checkpoint: C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\checkpoints\sweep_l2_0.pt
  sha256_16: ca6db04913814746
  n_input              700
  n_hidden             256
  n_output             20
  n_bins               700
  bin_ms               2.0
  readout_mode         sum
  neuron_param_mode    bias
  tau_mem_ms           20.0
  tau_syn_ms           5.0
  threshold            1.0
  config compatibility: OK


## 10. Evaluation settings (separate from construction)

Evaluation is a **separate operation**: it runs on PROBE against the *frozen* representation and
never touches the configuration that produced it. Targets come from the existing evaluation stack
(`neuron_zscored` is the rate/amplitude-independent diagnostic; `raw` includes the response level).

In [12]:
print("evaluation targets (existing stack)")
for name, description in sorted(panel.evaluation_targets().items()):
    print(f"  {name:18s} {description}")

# ---------------------------------------------------------------------------
EVALUATION_TARGET = state["evaluation_target"]   # <- EDITABLE ('all' or one target above)
EVALUATION_MODE = state["evaluation_mode"]       # <- EDITABLE
N_PERM = state["n_perm"]                         # <- EDITABLE (Mantel permutations)
BOOTSTRAP = state["bootstrap"]                   # <- EDITABLE (0 disables the CI)
N_SPLITS = state["n_splits"]                     # <- EDITABLE (CV folds over neurons)
# ---------------------------------------------------------------------------
try:
    panel.set_controls(state, evaluation_target=EVALUATION_TARGET, evaluation_mode=EVALUATION_MODE,
                       n_perm=N_PERM, bootstrap=BOOTSTRAP, n_splits=N_SPLITS)
except panel.PanelError as exc:
    print("[invalid] ", exc)

print(f"\nmode={state['evaluation_mode']}  target={state['evaluation_target']}  "
      f"n_perm={state['n_perm']}  bootstrap={state['bootstrap']}  n_splits={state['n_splits']}")
print("\n".join(panel.evaluation_notice()))

evaluation targets (existing stack)
  all                'all' = the four response variants (raw, neuron_centered, neuron_zscored, mean_rate)
  class_rate_20d     secondary (class-conditioned) target
  mean_rate          main response target
  neuron_centered    main response target
  neuron_zscored     main response target
  raw                main response target
  temporal           secondary (class-conditioned) target

mode=response  target=all  n_perm=2000  bootstrap=500  n_splits=5
Evaluation uses PROBE.
Representation construction remains FIT-only.
TEST is untouched.


## 11. Validate + resolved configuration preview (the notebook's dry run)

This cell calls the **real V2 configuration validation** and prints the resolved configuration.
It loads no data, trains nothing and evaluates nothing - it is the notebook equivalent of

```
uv run python scripts/v2_control_panel.py --preset <preset> --dry-run
```

In [13]:
sync_panel_settings()
resolved = None
try:
    resolved = panel.validate_state(state)        # the single validation path (V2Config + pipeline rules)
except panel.PanelError as exc:
    print("[INVALID CONFIGURATION]")
    print(exc)

if resolved is not None:
    print(f"VALID   |   run_id = {resolved.run_id}")
    print()
    print("\n".join(panel.preview_lines(resolved)))
    if resolved.warnings:
        print()
        print("The warnings above are advisory; hard errors were already raised.")

VALID   |   run_id = 7b71be5d019bab02

CHECKPOINT
  path                               C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\checkpoints\sweep_l2_0.pt
  sha256_16                          ca6db04913814746
  exists                             True
  run_id                             7b71be5d019bab02

MODEL
  model_type                         recurrent_lif
  n_hidden                           256
  n_layers                           1 (multi-layer not implemented)
  n_input / n_bins                   700 / 700 (2.0 ms bins)

FIT / DEV / PROBE / TEST POLICY
  FIT                                representation construction (record bank, residual training) only
  DEV                                never used by the control panel (model selection belongs to training)
  PROBE                              evaluation targets and metrics only
  TEST                               never accessed

TOTAL DIMENSION
  structured                         48
  residual               

## 12. Safety

The split discipline is fixed and displayed on every screen of this panel. There is no selectable
TEST option anywhere: the backend has no code path that opens the official test recordings.

In [14]:
print("split discipline")
print("================")
for line in panel.safety_lines():
    print("  " + line)
print()
print("  FIT   :", DATA_POLICY["fit"])
print("  DEV   :", DATA_POLICY["dev"])
print("  PROBE :", DATA_POLICY["probe"])
print("  TEST  :", DATA_POLICY["test"], "(official_test_loaded =", DATA_POLICY["official_test_loaded"], ")")
print()
print("Building a representation never evaluates it; evaluating never changes it.")

split discipline
  FIT:   representation construction + residual training
  PROBE: evaluation only
  TEST:  never accessed

  FIT   : representation construction (record bank, residual training) only
  DEV   : never used by the control panel (model selection belongs to training)
  PROBE : evaluation targets and metrics only
  TEST  : never accessed (official_test_loaded = False )

Building a representation never evaluates it; evaluating never changes it.


## 13. Build / load the representation (FIT only)

The only expensive cell. It runs the canonical backend build:

```
resolve (verified) -> FIT activity (cached) -> NeuronRecordBank -> structured encoder
                   -> residual (train or verified cache) -> NeuronVectors -> frozen artifact
```

If a compatible artifact for this `run_id` already exists it is **reused** (the cache is
validated: checkpoint, source schema + hash, dimensions, seed and the whole residual training
protocol). Setting `USE_CACHE = False` recomputes from scratch. Nothing here touches PROBE.

In [15]:
# ---------------------------------------------------------------------------
USE_CACHE = True        # <- EDITABLE: reuse verified cached artifacts instead of recomputing
# ---------------------------------------------------------------------------
import time

sync_panel_settings()
started = time.time()
build = panel.build_from_state(state, use_cache=USE_CACHE, verbose=True)
print()
for line in build.summary_lines():
    print(line)
print(f"\ntotal (this cell): {time.time() - started:.1f}s")

inventory = panel.cache_inventory(state=state)
print("\ncache")
print(f"  vector artifact : {inventory['vector_artifact']}  (exists: {inventory['vector_artifact_exists']})")
print(f"  resolved config : {inventory['resolved_config_exists']}")
print(f"  residual artifacts on disk: {inventory['n_residual_artifacts']}")
if build.residual_cache_path is not None:
    print(f"  residual for this run: {build.residual_cache_path}  (trained now: {build.residual_trained_now})")

[data] shd_train: 8156 samples, 63993588 events, layout=shd_ragged, speakers=yes
[cache] fit activity <- fit_activity_45980a6f4839f80a.npz

[build] run_id=7b71be5d019bab02
[build] representation shape: (256, 48)  [48 = 48 structured + 0 residual]
[build] record bank blocks available: intrinsic, input_conn, recurrent_in, recurrent_out, activity
[build] artifact: C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\results\neuron_vector_capacity\vectors\7b71be5d019bab02.npz
[build] fit activity cache: hit
[build] residual: none (learned_residual_d = 0)
[build] elapsed: 2.6s

total (this cell): 2.7s

cache
  vector artifact : C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\results\neuron_vector_capacity\vectors\7b71be5d019bab02.npz  (exists: True)
  resolved config : True
  residual artifacts on disk: 72


## 14. Inspect the frozen artifact

A pure artifact sanity check (no scientific evaluation): shape, coordinate groups, finiteness,
per-group statistics, provenance and the content hash.

In [16]:
info = panel.inspect_from_state(state)
print(f"run_id      : {info['run_id']}")
print(f"shape       : ({info['n_neurons']}, {info['d']})   "
      f"[structured {info['structured_d']} + residual {info['residual_d']}]")
print(f"coordinates : {info['n_coordinates']}   groups: {info['coordinate_groups']}")
print(f"finite      : {info['finite']}   (NaN: {info['n_nan']}, inf: {info['n_inf']})")
print(f"global      : mean {info['global_mean']:+.6f}   std {info['global_std']:.6f}")
print(f"per-coordinate mean: {info['column_mean']}")
print(f"per-coordinate std : {info['column_std']}")
print(f"matrix sha256: {info['matrix_sha256']}")
print(f"\nfirst coordinates: {info['feature_names_head']}")
print(f"last coordinates : {info['feature_names_tail']}")
print("\nper-group statistics")
for group, stats in info["per_group"].items():
    print(f"  {group:18s} n={stats['n_coordinates']:3d}  mean={stats['mean']:+.5f}  "
          f"std={stats['std']:.5f}  absmax={stats['absmax']:.5f}  "
          f"nan={stats['any_nan']} inf={stats['any_inf']}")

print("\nprovenance (structured encoder)")
for key, value in dict(info["provenance"].get("composition", {}).get("structured", {})).items():
    if key != "feature_names":
        print(f"  {key:22s} {value}")

if info["residual_source"] is not None:
    print("\nprovenance (learned residual)")
    for key, value in info["residual_source"].items():
        print(f"  {key:22s} {value}")
else:
    print("\nno learned residual in this representation (residual_d = 0)")

run_id      : 7b71be5d019bab02
shape       : (256, 48)   [structured 48 + residual 0]
coordinates : 48   groups: {'intrinsic': 1, 'input_conn': 14, 'recurrent_in': 19, 'recurrent_out': 14}
finite      : True   (NaN: 0, inf: 0)
global      : mean +4.373315   std 19.715148
per-coordinate mean: {'min': -0.0473837196442488, 'median': 0.17949863978794645, 'max': 134.25616442278272, 'mean': 4.373315170070485}
per-coordinate std : {'min': 0.001261825460753626, 'median': 0.020938304389447354, 'max': 4.352776437447021, 'mean': 0.19576729119038497}
matrix sha256: 56196a3bb640b7740968a16964a617f28ea6ab1f795c2c79250c38412a70b4bc

first coordinates: ['intrinsic.learned_bias', 'input_conn.entropy', 'input_conn.l1', 'input_conn.l2', 'input_conn.max_abs', 'input_conn.mean', 'input_conn.n_significant_frac', 'input_conn.neg_frac']
last coordinates : ['recurrent_out.neg_frac', 'recurrent_out.participation_ratio_frac', 'recurrent_out.pos_frac', 'recurrent_out.pos_neg_balance', 'recurrent_out.rms', 'recurr

## 15. Evaluate on PROBE (optional, explicit)

Separate from construction. It loads the frozen artifact (verified), builds the PROBE targets and
runs the **existing** evaluation stack (Mantel / prediction / controls) - the same numbers the CLI
produces. Evaluation settings cannot change the representation: the artifact is read-only here.

In [17]:
print("\n".join(panel.evaluation_notice()))
sync_panel_settings()
evaluation = panel.evaluate_from_state(state)      # loads the frozen artifact; PROBE targets only
print()
for line in evaluation.summary_lines():
    print(line)

print("\nrows")
for row in evaluation.rows:
    print({
        key: row.get(key)
        for key in ("target_variant", "value", "bootstrap_low", "bootstrap_high",
                    "permutation_p", "prediction_value")
    })
print(f"\nevaluation artifact: {evaluation.evaluation_path}")
print("the representation artifact was not modified:", evaluation.artifact_path)

Evaluation uses PROBE.
Representation construction remains FIT-only.
TEST is untouched.
[data] shd_train: 8156 samples, 63993588 events, layout=shd_ragged, speakers=yes

[evaluate] run_id=7b71be5d019bab02 mode=response target=all
[evaluate] artifact: C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\results\neuron_vector_capacity\vectors\7b71be5d019bab02.npz
[evaluate] statistics: n_perm=2000, bootstrap=500, n_splits=5, seed=0
[evaluate] results: C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\results\neuron_vector_capacity\evaluations\evaluation_ae10bcc47b4a086f.json
  target raw                mantel_spearman_r = +0.1126 CI[+0.030, +0.216] (p=0.005497251374312844)
  target neuron_centered    mantel_spearman_r = +0.1281 CI[+0.035, +0.247] (p=0.0009995002498750624)
  target neuron_zscored     mantel_spearman_r = +0.0039 CI[-0.011, +0.042] (p=0.383808095952024)
  target mean_rate          mantel_spearman_r = +0.0607 CI[+0.003, +0.142] (p=0.0384807596201899)
[evaluate] 

## 16. Run identity / metadata ("what exactly did I run?")

`run_id` is a hash of the **resolved representation values** (component schema versions, the vector
configuration including the whole residual protocol, the dtypes and the checkpoint id). Evaluation
settings, the preset label and the wall clock deliberately do not enter it, so the same
configuration always maps to the same artifact. The build cell writes the resolved configuration
next to the artifact; this cell shows where it is.

In [18]:
if resolved is not None:
    print(f"run_id            : {resolved.run_id}")
    print(f"resolved config   : {resolved.resolved_config_path}")
    print(f"vector artifact   : {resolved.vector_artifact_path}")
    print(f"checkpoint        : {resolved.checkpoint_path}  (sha256_16={resolved.checkpoint_sha256_16})")
    print(f"overrides applied : {list(resolved.cli_overrides) or '(none: the preset alone)'}")
    print("\npanel state (serialisable)")
    import json

    print(json.dumps(panel.state_export(state), indent=2, default=str))
else:
    print("run the validation cell first (section 11)")

run_id            : 7b71be5d019bab02
resolved config   : C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\results\neuron_vector_capacity\resolved_configs\7b71be5d019bab02.json
vector artifact   : C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\results\neuron_vector_capacity\vectors\7b71be5d019bab02.npz
checkpoint        : C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\checkpoints\sweep_l2_0.pt  (sha256_16=ca6db04913814746)
overrides applied : (none: the preset alone)

panel state (serialisable)
{
  "schema": "v2_panel_state/v1",
  "preset": "historical_48",
  "config_path": "configs/neuron_space_baseline.yaml",
  "checkpoint": "C:\\Users\\Administrator\\Desktop\\compneuro\\neuron_as_vector\\checkpoints\\sweep_l2_0.pt",
  "out_dir": "results/neuron_vector_capacity",
  "controls": {
    "config_path": "configs/neuron_space_baseline.yaml",
    "preset": "historical_48",
    "checkpoint": "C:\\Users\\Administrator\\Desktop\\compneuro\\neuron_as_vector\\checkpo

## 17. Reproducibility block

Copy/paste-friendly: everything needed to reproduce this configuration from the command line.
This complements (and does not replace) the CLI summary mechanism.

In [19]:
if resolved is not None:
    print("\n".join(panel.reproducibility_lines(resolved, extra={"panel state schema": panel.PANEL_SCHEMA})))
else:
    print("run the validation cell first (section 11)")

REPRODUCIBILITY
  run_id                : 7b71be5d019bab02
  config                : C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\configs\neuron_space_baseline.yaml
  preset                : historical_48
  cli-equivalent        : uv run python scripts/v2_control_panel.py --config C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\configs\neuron_space_baseline.yaml --preset historical_48
  checkpoint            : C:\Users\Administrator\Desktop\compneuro\neuron_as_vector\checkpoints\sweep_l2_0.pt (sha256_16=ca6db04913814746)
  dimensions            : d=48 = 48 structured + 0 residual
  blocks                : intrinsic, input_conn, recurrent_in, recurrent_out
  residual seeds        : seed=0, split_seed=0, mask_seed=0
  schemas               : pipeline=v2_pipeline/v1, components={'record_bank': 'neuron_record_bank/v1', 'structured_vector': 'structured_vector/v1', 'residual_source': 'residual_source/v1', 'learned_residual': 'learned_residual/v1', 'neuron_vector': 'neu

## 18. Parent summary (identical to the CLI)

The same machine-readable project-state block the CLI prints. The CLI command remains the
canonical headless version:

```
uv run python scripts/v2_control_panel.py --summary-for-parent [--run-tests]
```

In [20]:
print(parent_summary(resolved, include_tests=False))
print()
print("test count: run `uv run pytest -q`, or the CLI with --run-tests for the recorded result.")

=== V2 PARENT SUMMARY ===

STATUS:
Representation architecture frozen; one backend (src/v2_pipeline.py) with two interfaces - the Jupyter control panel (notebooks/V2_Control_Panel.ipynb, primary interactive) and the CLI (scripts/v2_control_panel.py, headless/reproducible); no new representation source, no network_context, no multi-layer support, TEST never used.

TESTS:
unknown (pytest could not be invoked in this environment)

ACTIVE DOCS:
  README.md, VECTOR_V2_AUDIT.md, V2_STATUS.md

CORE MODULES:
  src/v2_config.py (single configuration system)
  src/neuron_record.py (NeuronRecordBank)
  src/structured_vector.py (StructuredVectorEncoder)
  src/functional_response.py (label-free functional-response source)
  src/residual.py (learned residual)
  src/neuron_vector.py (composition + frozen artifact)
  src/v2_pipeline.py (stable API + presets + run identity)
  src/v2_panel.py (notebook-facing control-surface state layer)
  src/vector_capacity.py / rate_robustness.py / source_extension.p

## 19. Workflow recipes

Change **one control** and follow the same four steps (validate → build → inspect → optionally
evaluate):

```
historical_48  ->  STRUCTURED_D = 64                -> validate -> build -> inspect
functional_64  ->  RESIDUAL_SEED = 1                -> validate -> build -> evaluate
functional_64  ->  RESIDUAL_EPOCHS = 50 (drift!)    -> validate -> build -> inspect
structural + activity: ENABLED_BLOCKS += "activity" -> validate -> build -> inspect
temporal deterministic: PRESET = "temporal_structured" -> validate -> build -> inspect
```

Rules of thumb

* change `LEARNED_RESIDUAL_D` and `RESIDUAL_ENABLED` together (the check in section 3 reports it
  immediately if they disagree);
* the functional-response / temporal **residual sources** need the residual; the temporal
  **encoder block** does not;
* anything that changes the resolved representation changes the `run_id` and therefore the cached
  artifact - that is intended, so two configurations can never share an artifact by accident;
* nothing here launches a sweep: one state, one build, one evaluation at a time.

## 20. Notes and limitations

* **Frozen architecture.** This panel configures the frozen V2 representation; it is not a place to
  add sources. `network_context` and multi-layer SNNs are **not implemented** and are not
  selectable.
* **No hidden tuning.** Nothing in the panel selects a dimension, normalisation, seed or mask mode
  from PROBE results; it is an interface, not an optimiser.
* **Small effects, honest reporting.** The latest evaluations found modest correspondences, a small
  but reproducible contribution of the functional-response source on the rate/amplitude-independent
  target, and no established "best" dimension or normalisation. See `V2_STATUS.md` §11-§13.
* **Training protocol.** Keep the established residual protocol for comparable artefacts; the drift
  line in section 7 tells you when you are not doing so.
* **Environment.** This notebook needs a Jupyter kernel but installs nothing. `ipywidgets` is
  optional. For headless/automated use, the same configuration is reproducible through
  `scripts/v2_control_panel.py`.